In [2]:
with open ('input.txt', 'r') as file:
    text = file.read()

In [3]:
print("len of dataset in characters:", len(text))

len of dataset in characters: 1115394


In [4]:
print("First 1000 characters:")
print(text[:1000])

First 1000 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not i

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

#65 characters in this entire dataset 


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


# Encoding Into Integers (building the encoder and the decoder)

In [ ]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # take a string and return a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # take a list of integers and return a string

print(encode("hello world"))
print(decode(encode("hello world")))

#use BPE in actual implementation

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [8]:
import torch 

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

### Train test split

In [9]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
#Train in chunks, maximum length is called block_size
block_size = 8
train_data[:block_size+1] # first 9 characters
#simulatenously predict the next character in each element.

#thats why its +1, because now there's 8 examples. 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size] # first 8 characters
y = train_data[1:block_size+1] # next 8 characters
for t in range(block_size):
    context = x[:t+1] # first t+1 characters
    target = y[t] # next character
    print(f"When input is {context}, the target is {target}.")  

When input is tensor([18]), the target is 47.
When input is tensor([18, 47]), the target is 56.
When input is tensor([18, 47, 56]), the target is 57.
When input is tensor([18, 47, 56, 57]), the target is 58.
When input is tensor([18, 47, 56, 57, 58]), the target is 1.
When input is tensor([18, 47, 56, 57, 58,  1]), the target is 15.
When input is tensor([18, 47, 56, 57, 58,  1, 15]), the target is 47.
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is 58.


In [12]:
#Now batch size, we process multiple chunks at the same time. 
torch.manual_seed(1337) # for reproducibility
batch_size = 4 # how many independent sequences will we process in parallel 
block_size = 8 # how many characters to predict in parallel

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # random starting points for each sequence
    x = torch.stack([data[i:i+block_size] for i in ix]) # batch of input sequences
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # batch of target sequences
    return x, y

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)
print("---")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1] # first t+1 characters of the b-th sequence
        target = yb[b, t] # next character of the b-th sequence
        print(f"When input is {context}, the target is {target}.")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---
When input is tensor([24]), the target is 43.
When input is tensor([24, 43]), the target is 58.
When input is tensor([24, 43, 58]), the target is 5.
When input is tensor([24, 43, 58,  5]), the target is 57.
When input is tensor([24, 43, 58,  5, 57]), the target is 1.
When input is tensor([24, 43, 58,  5, 57,  1]), the target is 46.
When input is tensor([24, 43, 58,  5, 57,  1, 46]), the target is 43.
When input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), the target is 39.
When input is tensor([44]), the target is 53.
When input is tensor([44, 53]), the target is 56.
When input is tensor([44, 53,